# Detecção de Fraude em Cartão de Crédito — Pipeline v2.1

**Estudo comparado com validação temporal, ablação de balanceamento, decisão por custo econômico e inferência estatística**

---

Este notebook reorganiza e corrige a metodologia da versão original do TCC. Cada seção traz: **(i)** a explicação do conceito, **(ii)** por que a etapa é necessária, **(iii)** a referência bibliográfica que a fundamenta, e **(iv)** o código comentado linha a linha.

### O que este pipeline corrige em relação à v1

| # | Problema na v1 | Correção aqui | Referência |
|---|---|---|---|
| 1 | Threshold otimizado no conjunto de **teste** (vazamento de seleção) | Thresholds escolhidos **só na validação** e congelados | Cawley & Talbot (2010) |
| 2 | Split aleatório em dados temporais; `Time` como feature | Split **out-of-time** 70/15/15; `Time` → `hour` | Dal Pozzolo (2015) |
| 3 | `class_weight` sempre ligado **+** SMOTE por cima (efeitos confundidos) | **Ablação**: baseline / class_weight / SMOTE, um fator por vez | He & Garcia (2009) |
| 4 | ROC-AUC como métrica-manchete sob prevalência de 0,17% | **PR-AUC (Average Precision)** como métrica primária | Saito & Rehmsmeier (2015) |
| 5 | F1 pesa FP e FN igualmente (escolha economicamente arbitrária) | Threshold por **custo esperado em R$** (FN = valor da transação) | Elkan (2001); Bahnsen et al. (2014) |
| 6 | `decision_function` do SVM tratada como probabilidade | **Calibração de Platt** via `CalibratedClassifierCV` | Platt (1999) |
| 7 | 1 split, 1 seed, nenhum teste estatístico | **5 seeds** + **bootstrap IC 95%** + **McNemar** | Dietterich (1998) |
| 8 | Sem gradient boosting nem métodos não supervisionados | **XGBoost**, Isolation Forest, **Autoencoder** | Chen & Guestrin (2016); Liu et al. (2008) |
| 9 | PCA reaplicado sobre variáveis que já são PCA | Sem PCA; **SHAP** para interpretabilidade | Lundberg & Lee (2017) |
| 10 | "Validação cruzada" anunciada e nunca executada | Tuning do XGBoost **avaliado na validação** | Cawley & Talbot (2010) |

> **Regra de ouro deste notebook**: o conjunto de teste é tocado **uma única vez**, na Seção 10, com todas as decisões já congeladas. Nenhuma célula anterior a ela lê `y_test` para tomar qualquer decisão.


## 0. Instalação das dependências

Execute esta célula **uma vez** por ambiente (Colab, Replit, Jupyter local ou VS Code). O `%pip` garante a instalação no mesmo kernel do notebook — `!pip` pode instalar em outro interpretador (causa clássica de `ModuleNotFoundError` mesmo após "instalar").

O `shap` é opcional (Seção 13): se a instalação falhar no seu ambiente, o notebook segue normalmente e apenas pula a etapa de interpretabilidade.

In [ ]:
# ===== INSTALAÇÃO DE TODOS OS MÓDULOS (executar uma vez) =====
# Núcleo científico: manipulação de dados, álgebra e gráficos
%pip install -q pandas numpy matplotlib scipy

# Modelos clássicos, métricas, calibração e pré-processamento
%pip install -q scikit-learn

# SMOTE dentro de Pipeline (evita vazamento — ver Seção 6)
%pip install -q imbalanced-learn

# Gradient boosting de referência em dados tabulares (CHEN; GUESTRIN, 2016)
%pip install -q xgboost

# OPCIONAL: interpretabilidade por valores de Shapley (LUNDBERG; LEE, 2017)
%pip install -q shap

print("Dependências instaladas. Reinicie o kernel se algum import falhar abaixo.")

## 1. Importações

Importamos tudo em um único bloco para que qualquer módulo ausente falhe **aqui**, e não no meio do experimento (que pode levar horas). Note a ausência de `warnings.filterwarnings('ignore')`: na v1, suprimir todos os avisos globalmente escondia problemas reais (ex.: divisão por zero no cálculo de melhoria de recall). Avisos são diagnóstico, não ruído.

In [ ]:
# ===== BIBLIOTECAS PADRÃO =====
import json                      # salvar resultados estruturados (McNemar)
import time                      # cronometrar treinos
from pathlib import Path         # caminhos de arquivo portáveis
from dataclasses import dataclass, asdict   # linhas da tabela de resultados

# ===== NÚCLEO CIENTÍFICO =====
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Teste binomial exato — base do teste de McNemar (Seção 8)
from scipy.stats import binomtest

# ===== MODELOS SUPERVISIONADOS =====
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import (RandomForestClassifier,          # bagging (BREIMAN, 2001)
                              HistGradientBoostingClassifier,  # boosting nativo do sklearn
                              IsolationForest)                 # anomalia (LIU et al., 2008)
from xgboost import XGBClassifier                              # boosting (CHEN; GUESTRIN, 2016)

# Autoencoder denso via MLP que aprende a reconstruir a própria entrada
from sklearn.neural_network import MLPRegressor

# ===== CALIBRAÇÃO E PRÉ-PROCESSAMENTO =====
from sklearn.calibration import CalibratedClassifierCV, CalibrationDisplay  # (PLATT, 1999)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline as SkPipeline

# Pipeline do imblearn: o SMOTE só roda no .fit (treino), nunca no .predict
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE                       # (CHAWLA et al., 2002)

# ===== MÉTRICAS =====
from sklearn.metrics import (average_precision_score,   # PR-AUC: métrica primária
                             roc_auc_score,             # secundária (inflada em desbalanceamento)
                             brier_score_loss,          # qualidade da calibração
                             precision_recall_curve,
                             precision_score, recall_score, f1_score,
                             confusion_matrix)

print("Imports OK — sklearn", __import__("sklearn").__version__,
      "| xgboost", __import__("xgboost").__version__)

## 2. Configuração do experimento

Todas as escolhas do estudo ficam **explícitas e em um só lugar** — reprodutibilidade começa aqui. Dois parâmetros merecem atenção:

- **`cost_fp`**: custo, em R$, de bloquear uma transação legítima (revisão manual + atrito com o cliente). É a variável que conecta o experimento à **Teoria dos Custos de Transação** (WILLIAMSON, 1998) usada na fundamentação da monografia: o classificador é um mecanismo de governança que minimiza custos ex-post de contratos incompletos, e o threshold ótimo é uma decisão econômica (ELKAN, 2001). Como esse custo é uma estimativa, a Seção 12 faz **análise de sensibilidade** sobre `cost_fp_grid`.
- **`n_seeds`**: número de sementes aleatórias por configuração estocástica. Na v1, a Árvore de Decisão — que o próprio texto descrevia como instável — era avaliada com **uma** execução. Aqui, reportamos média ± desvio entre seeds (Seção 8).

> Para uma primeira passada rápida, use `sample_n = 100_000` e `n_seeds = 3`. **Decida o orçamento antes de ver qualquer resultado do teste** — reduzir depois de olhar seria reintroduzir o vazamento de seleção (CAWLEY; TALBOT, 2010).

In [ ]:
# ===== CONFIGURAÇÃO CENTRAL DO EXPERIMENTO =====
CONFIG = {
    "data_path": "creditcard.csv",   # dataset Kaggle mlg-ulb/creditcardfraud
    "output_dir": "outputs_v2_1",    # onde salvar tabelas e figuras

    "base_seed": 42,                 # semente-base (reprodutibilidade)
    "n_seeds": 5,                    # execuções por configuração estocástica

    # Split temporal (ordenado por 'Time'): passado -> presente -> futuro
    "train_frac": 0.70,              # 70% mais antigo: TREINO
    "val_frac": 0.15,                # 15% seguinte:    VALIDAÇÃO (todas as decisões)
                                     # 15% restante:    TESTE (tocado uma única vez)

    # Custos econômicos (R$) — ver derivação do threshold ótimo na Seção 7
    "cost_fp": 10.0,                 # custo de revisão manual por falso positivo
    "cost_fp_grid": [2.0, 10.0, 50.0],  # grade da análise de sensibilidade

    "n_bootstrap": 500,              # reamostragens p/ IC 95% (Seção 8)
    "sample_n": None,                # ex.: 100_000 para teste rápido; None = completo
    "run_shap": True,                # Seção 13 (pulada se shap não instalado)
    "run_xgb_tuning": True,          # Seção 9 (seleção na validação)
}

# Modelos supervisionados avaliados (os 4 do TCC original + 3 modernos)
SUPERVISED = ["Logistic Regression", "Linear SVM (calibrated)", "Decision Tree",
              "Gaussian NB", "Random Forest", "Hist Gradient Boosting", "XGBoost"]

# Detectores de anomalia (não usam o rótulo no treino — Seção 6.2)
UNSUPERVISED = ["Isolation Forest", "Autoencoder"]

# Estratégias de balanceamento em ABLAÇÃO (mutuamente exclusivas — Seção 6)
STRATEGIES = ["baseline", "class_weight", "smote"]

# Modelos deterministas dado o dado (repetir seeds seria desperdício):
DETERMINISTIC = {"Logistic Regression", "Linear SVM (calibrated)", "Gaussian NB"}

OUTDIR = Path(CONFIG["output_dir"]); OUTDIR.mkdir(exist_ok=True)
print("Configuração carregada. Saídas em:", OUTDIR.resolve())

## 3. Carregamento dos dados

O dataset contém 284.807 transações de portadores europeus em setembro de 2013, das quais 492 (0,1728%) são fraudes. As variáveis V1–V28 **já são componentes principais** (PCA aplicado na origem por anonimização) — motivo pelo qual este pipeline **não reaplica PCA**, ao contrário da v1: reaplicar PCA sobre componentes ortogonais é redundante (reduz 30 → ~28 dimensões) e destrói a pouca interpretabilidade restante.

**Citação correta do dataset** (a v1 citava "KAGGLE, 2010", que não existe): DAL POZZOLO, A.; CAELEN, O.; JOHNSON, R. A.; BONTEMPI, G. *Calibrating Probability with Undersampling for Unbalanced Classification*. IEEE SSCI, 2015 — grupo MLG da Université Libre de Bruxelles.

In [ ]:
def carregar_dados(caminho: str) -> pd.DataFrame:
    """Carrega o creditcard.csv com dtypes explícitos e valida o schema."""
    # dtypes explícitos: economia de memória e detecção precoce de arquivo errado
    dtypes = {"Time": "float64", "Amount": "float64", "Class": "int8"}
    dtypes.update({f"V{i}": "float32" for i in range(1, 29)})

    df = pd.read_csv(caminho, dtype=dtypes)

    # Validação do schema: falhar AQUI, com mensagem clara, e não no meio do treino
    esperadas = {"Time", "Amount", "Class"} | {f"V{i}" for i in range(1, 29)}
    faltando = esperadas - set(df.columns)
    if faltando:
        raise ValueError(f"Colunas ausentes no CSV: {sorted(faltando)}")
    return df


df = carregar_dados(CONFIG["data_path"])
if CONFIG["sample_n"]:
    # Amostra rápida = primeiras N transações NO TEMPO (preserva a ordem temporal;
    # um .sample() aleatório destruiria a estrutura que a Seção 5 protege)
    df = df.sort_values("Time").head(CONFIG["sample_n"])

taxa = df["Class"].mean()
print(f"{len(df):,} transações | {int(df['Class'].sum())} fraudes "
      f"({taxa*100:.4f}%) | desbalanceamento {(1-taxa)/taxa:.0f}:1")
df.head(3)

## 4. Engenharia de features

Duas transformações, ambas justificadas — e uma **remoção** importante:

1. **`hour`** = hora do dia extraída de `Time`. Fraude tem padrão circadiano documentado (madrugada concentra atividade fraudulenta); a *hora do dia* é um sinal cíclico que **generaliza para o futuro**.
2. **`log_amount`** = `log(1 + Amount)`. A distribuição de valores tem cauda pesada; o log estabiliza a escala para os modelos lineares (Regressão Logística, SVM).
3. **`Time` bruto é REMOVIDO da matriz de features.** Em split temporal ele é um índice monotônico: o teste contém apenas valores *maiores* que todos os do treino, forçando extrapolação sem significado. Na v1 (split aleatório), era pior: permitia ao modelo memorizar a *vizinhança temporal* das fraudes do teste que vazaram para o treino.

O `Amount` **bruto** é preservado fora de X: ele é o custo do falso negativo na função de perda econômica (Seção 7).

In [ ]:
def engenharia_features(df: pd.DataFrame) -> pd.DataFrame:
    """Cria 'hour' e 'log_amount'; mantém 'Time' e 'Amount' brutos apenas
    para ordenação temporal e cálculo de custo, respectivamente."""
    out = df.copy()
    out["hour"] = (out["Time"] // 3600) % 24     # segundos -> hora do dia (0-23)
    out["log_amount"] = np.log1p(out["Amount"])  # log(1+x): definido em Amount=0
    return out


df = engenharia_features(df)

# Matriz de features: V1-V28 (PCA de origem) + as duas engenheiradas
FEATURES = [f"V{i}" for i in range(1, 29)] + ["hour", "log_amount"]
print(f"{len(FEATURES)} features:", FEATURES[:3], "...", FEATURES[-2:])

## 5. Split temporal (out-of-time)

**Por que não split aleatório?** Fraude evolui no tempo (*concept drift* — GAMA et al., 2014): fraudadores adaptam táticas, e o uso real do modelo é sempre *prever o futuro com o passado*. O protocolo de avaliação deve imitar isso: **treino = passado, validação = presente, teste = futuro**. O split aleatório estratificado da v1 deixava transações "do futuro" no treino, inflando as métricas — e a Discussão da v1 alegava "ausência de dados temporais" para não tratar drift, quando a coluna `Time` estava no dataset (DAL POZZOLO, 2015, constrói a tese inteira sobre essa estrutura de *stream*).

**Papel de cada partição:**
- **Treino (70%)** — ajusta modelos, scaler, SMOTE, calibração.
- **Validação (15%)** — recebe **todas as decisões**: hiperparâmetros do XGBoost, thresholds de F1 e de custo, escolha de estratégia por modelo.
- **Teste (15% mais recente)** — tocado **uma única vez** na Seção 10, com tudo congelado.

Note que **não estratificamos**: forçar a mesma taxa de fraude em cada partição destruiria a temporalidade. Prevalências diferentes entre partições são realismo, não defeito.

In [ ]:
def split_temporal(df: pd.DataFrame):
    """Ordena por 'Time' e corta em treino/validação/teste consecutivos.
    Devolve, por partição: (X, y, amount) — amount fica FORA de X e
    alimenta a função de custo (Seção 7)."""
    df = df.sort_values("Time").reset_index(drop=True)   # garante ordem temporal
    n = len(df)
    i_tr = int(n * CONFIG["train_frac"])                              # fim do treino
    i_va = int(n * (CONFIG["train_frac"] + CONFIG["val_frac"]))       # fim da validação

    parts = {}
    for nome, sl in (("train", slice(0, i_tr)),
                     ("val",   slice(i_tr, i_va)),
                     ("test",  slice(i_va, n))):
        sub = df.iloc[sl]
        parts[nome] = (sub[FEATURES].to_numpy(np.float64),   # X
                       sub["Class"].to_numpy(np.int8),       # y
                       sub["Amount"].to_numpy(np.float64))   # valor (p/ custo)
        y = parts[nome][1]
        print(f"  {nome:5s}: {len(y):>7,} obs | fraudes: {int(y.sum()):>4} "
              f"({y.mean()*100:.3f}%)")
    return parts


parts = split_temporal(df)
X_tr, y_tr, amt_tr = parts["train"]
X_va, y_va, amt_va = parts["val"]
X_te, y_te, amt_te = parts["test"]

# Razão negativos/positivos do TREINO: alimenta scale_pos_weight do XGBoost
POS_RATIO = float((y_tr == 0).sum() / max(1, (y_tr == 1).sum()))
print(f"\nRazão neg/pos no treino (p/ scale_pos_weight): {POS_RATIO:.0f}:1")

## 6. Modelos e estratégias de balanceamento — desenho em **ablação**

**O erro de identificação da v1**: todos os modelos tinham `class_weight='balanced'` *sempre ligado*, o SMOTE era aplicado *por cima*, e o threshold era ajustado *por cima de tudo*. Três mecanismos que atacam o mesmo problema, empilhados — o efeito de cada um fica **não identificável** (em linguagem econométrica: colinearidade perfeita entre tratamentos). O achado da v1 de que "SMOTE não melhorou o F1" era inaproveitável, pois o baseline nunca esteve realmente desbalanceado do ponto de vista da função de perda.

**Aqui, um fator por vez** (HE; GARCIA, 2009):
- **`baseline`** — nenhuma correção;
- **`class_weight`** — reponderação da classe minoritária na função de perda (no XGBoost: `scale_pos_weight = N_neg/N_pos`; no Naive Bayes, o análogo é `priors=[0.5, 0.5]`);
- **`smote`** — oversampling sintético por interpolação de vizinhos (CHAWLA et al., 2002), **dentro de um `imblearn.Pipeline`**: o passo roda apenas no `.fit()` (treino) e nunca contamina validação/teste. A v1 descrevia no texto o SMOTE "antes da divisão" — que seria vazamento clássico.

**Detalhes por modelo:**
- `LinearSVC` não produz probabilidade (`decision_function` é distância ao hiperplano) — envolvemos em `CalibratedClassifierCV(method='sigmoid')`, a **calibração de Platt** (PLATT, 1999), ajustada por CV interna *no treino*.
- `StandardScaler` só nos modelos sensíveis a escala (lineares), e **dentro do Pipeline** — na v1 ele era ajustado fora, no dataset já dividido, mas a disciplina de Pipeline garante o comportamento correto também sob validação cruzada.
- O XGBoost usa `eval_metric='aucpr'` — coerente com a métrica primária do estudo (Seção 8).

In [ ]:
def make_supervised(nome: str, estrategia: str, seed: int):
    """Fábrica de pipelines supervisionados.
    Garante a ABLAÇÃO: 'class_weight' ativa reponderação NO modelo;
    'smote' ativa oversampling NO pipeline (e desativa reponderação);
    'baseline' não ativa nada."""
    cw = "balanced" if estrategia == "class_weight" else None

    if nome == "Logistic Regression":
        est = LogisticRegression(max_iter=5000,          # garante convergência
                                 class_weight=cw,
                                 solver="lbfgs",
                                 random_state=seed)
        precisa_scaler = True                            # modelo linear: sensível a escala

    elif nome == "Linear SVM (calibrated)":
        base = LinearSVC(class_weight=cw, dual=False,    # dual=False: n_amostras >> n_features
                         max_iter=10000, random_state=seed)
        # decision_function NÃO é probabilidade -> calibração de Platt (sigmóide),
        # ajustada por CV=3 exclusivamente dentro do TREINO (PLATT, 1999)
        est = CalibratedClassifierCV(base, method="sigmoid", cv=3)
        precisa_scaler = True

    elif nome == "Decision Tree":
        est = DecisionTreeClassifier(max_depth=8,        # controla sobreajuste
                                     min_samples_leaf=50,
                                     class_weight=cw, random_state=seed)
        precisa_scaler = False                           # árvores são invariantes a escala

    elif nome == "Gaussian NB":
        # Análogo de class_weight p/ NB: priors uniformes anulam o prior desbalanceado
        est = GaussianNB(priors=[0.5, 0.5] if estrategia == "class_weight" else None)
        precisa_scaler = False

    elif nome == "Random Forest":
        # Bagging (BREIMAN, 2001): árvores profundas independentes; média reduz variância
        est = RandomForestClassifier(n_estimators=200, min_samples_leaf=5,
                                     class_weight=cw, n_jobs=-1, random_state=seed)
        precisa_scaler = False

    elif nome == "Hist Gradient Boosting":
        # Boosting por histogramas nativo do sklearn (mesma família do LightGBM)
        est = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.1,
                                             class_weight=cw, random_state=seed)
        precisa_scaler = False

    elif nome == "XGBoost":
        # Gradient boosting regularizado (CHEN; GUESTRIN, 2016).
        # Hiperparâmetros vêm do tuning da Seção 9 (escolhidos NA VALIDAÇÃO).
        est = XGBClassifier(
            n_estimators=CONFIG.get("_xgb_n_estimators", 400),
            learning_rate=CONFIG.get("_xgb_lr", 0.1),
            max_depth=CONFIG.get("_xgb_depth", 6),
            subsample=0.8, colsample_bytree=0.8,     # aleatorização anti-sobreajuste
            # scale_pos_weight: mecanismo canônico do XGB p/ desbalanceamento
            scale_pos_weight=POS_RATIO if estrategia == "class_weight" else 1.0,
            tree_method="hist", eval_metric="aucpr",  # coerente com a métrica primária
            n_jobs=-1, random_state=seed)
        precisa_scaler = False
    else:
        raise ValueError(nome)

    # Montagem do pipeline na ordem: [SMOTE] -> [Scaler] -> Modelo
    passos = []
    if estrategia == "smote":
        # k_neighbors=5 é o padrão de CHAWLA et al. (2002).
        # Dentro do Pipeline, o SMOTE roda APENAS no .fit -> sem vazamento.
        passos.append(("smote", SMOTE(random_state=seed, k_neighbors=5)))
    if precisa_scaler:
        passos.append(("scaler", StandardScaler()))
    passos.append(("clf", est))
    return ImbPipeline(passos) if estrategia == "smote" else SkPipeline(passos)


print("Fábrica de modelos supervisionados pronta:", len(SUPERVISED), "modelos x",
      len(STRATEGIES), "estratégias.")

### 6.2 Detectores de anomalia (não supervisionados)

Estes modelos respondem a uma pergunta de negócio **diferente**: *e quando o padrão de fraude é novo, sem rótulo no histórico?* — exatamente o cenário de concept drift. A conclusão da v1 recomendava "aprendizado não supervisionado" para trabalhos futuros sem executar nenhum método; aqui rodamos os dois canônicos:

- **Isolation Forest** (LIU; TING; ZHOU, 2008): pontos raros são fáceis de *isolar* com cortes aleatórios; o score é a suscetibilidade ao isolamento. Treina em todo o treino **sem olhar o rótulo**.
- **Autoencoder** (GOODFELLOW et al., 2016, cap. 14): rede 30→16→8→16→30 treinada para reconstruir a própria entrada, **apenas nas transações legítimas** — aprende a "variedade do normal"; a fraude, vinda de outra distribuição, reconstrói mal, e o **erro de reconstrução é o score de anomalia**. É a materialização concreta do "deep learning" que a v1 invocava como promessa vaga.

**Protocolo de comparação (semi-supervisionado padrão):** os rótulos da *validação* são usados somente para escolher o threshold; os scores são normalizados para [0,1] pelos limites **da validação** (nem a normalização pode olhar o teste). Como o score normalizado não é probabilidade calibrada, o Brier score é omitido para esses modelos.

In [ ]:
class IsoForestScorer:
    """Isolation Forest (LIU et al., 2008): fit sem rótulo; score = anomalia."""

    def __init__(self, seed: int):
        self.model = IsolationForest(n_estimators=200,
                                     max_samples=0.5,        # subamostra por árvore
                                     contamination="auto",
                                     n_jobs=-1, random_state=seed)

    def fit(self, X, y=None):
        self.model.fit(X)                # y ignorado de propósito: não supervisionado
        return self

    def score(self, X):
        # score_samples: maior = mais normal; invertemos p/ maior = mais anômalo
        return -self.model.score_samples(X)


class AutoencoderScorer:
    """Autoencoder denso 30->16->8->16->30 via MLPRegressor (X -> X),
    treinado SOMENTE nas transações legítimas do treino.
    Score = erro quadrático médio de reconstrução por transação."""

    def __init__(self, seed: int):
        self.scaler = StandardScaler()          # redes exigem entradas padronizadas
        self.ae = MLPRegressor(hidden_layer_sizes=(16, 8, 16),  # gargalo de 8 dims
                               activation="relu",
                               max_iter=60,
                               early_stopping=True,   # para quando a validação interna estagna
                               n_iter_no_change=5,
                               random_state=seed)

    def fit(self, X, y):
        X_leg = X[y == 0]                        # APENAS legítimas: aprende o "normal"
        Xs = self.scaler.fit_transform(X_leg)
        self.ae.fit(Xs, Xs)                      # alvo = a própria entrada
        return self

    def score(self, X):
        Xs = self.scaler.transform(X)
        rec = self.ae.predict(Xs)                # reconstrução
        return np.mean((Xs - rec) ** 2, axis=1)  # MSE por linha = score de anomalia


def minmax_to_proba(scores_val: np.ndarray, scores: np.ndarray) -> np.ndarray:
    """Normaliza scores de anomalia p/ [0,1] usando limites da VALIDAÇÃO
    (nem a normalização pode olhar o teste). Não é probabilidade calibrada."""
    lo, hi = scores_val.min(), scores_val.max()
    return np.clip((scores - lo) / (hi - lo + 1e-12), 0, 1)


print("Detectores de anomalia prontos:", UNSUPERVISED)

## 7. A função de perda econômica e a escolha do threshold

**O ponto central da correção — e a ponte com a Economia.** O F1-Score, métrica-manchete da v1, pesa falso positivo e falso negativo **igualmente**: uma escolha economicamente arbitrária jamais justificada no texto. Mas os custos reais são assimétricos e conhecidos em estrutura:

- **Falso negativo** (fraude aprovada): perda ≈ **valor da transação** (chargeback) → custo *dependente do exemplo*;
- **Falso positivo** (legítima bloqueada): custo de **revisão manual + atrito** ≈ `cost_fp` fixo.

Pela teoria da decisão bayesiana (ELKAN, 2001), com probabilidade calibrada $p$ e custos $C_{FP}$, $C_{FN}$, a regra ótima é bloquear quando:

$$ p \;>\; \frac{C_{FP}}{C_{FP} + C_{FN}} $$

Com custo de FN dependente do valor, minimizamos empiricamente a **perda esperada total** na validação (formulação *example-dependent cost-sensitive* — BAHNSEN; AOUADA; OTTERSTEN, 2014, aplicada a este mesmo dataset):

$$ \text{Custo}(t) \;=\; \underbrace{\sum_{i \in FN(t)} \text{Amount}_i}_{\text{fraudes perdidas}} \;+\; \underbrace{|FP(t)| \cdot C_{FP}}_{\text{revisões manuais}} $$

Isso conecta a fundamentação teórica da monografia (custos de transação — WILLIAMSON, 1998) ao método: o resultado deixa de ser "F1 = 0,82" e passa a ser "**o sistema economiza R$ X no período versus aprovar tudo**".

**Regra inegociável:** ambos os thresholds (máx. F1 e mín. custo) são escolhidos **exclusivamente na validação** e depois congelados. Na v1, o threshold era otimizado no teste — o vazamento que invalidava o resultado de 0,8247.

In [ ]:
def threshold_max_f1(y_val, p_val) -> float:
    """Threshold que maximiza F1 NA VALIDAÇÃO (nunca no teste)."""
    prec, rec, thr = precision_recall_curve(y_val, p_val)
    f1 = 2 * prec * rec / (prec + rec + 1e-12)   # média harmônica ponto a ponto
    # precision_recall_curve devolve len(thr) = len(prec) - 1; alinhamos com [:-1]
    return float(thr[int(np.argmax(f1[:-1]))])


def custo_esperado(y_true, y_pred, amount, cost_fp: float) -> float:
    """Perda total em R$: soma dos valores das fraudes perdidas (FN)
    + custo fixo de revisão por transação legítima bloqueada (FP)."""
    fn = (y_true == 1) & (y_pred == 0)           # fraudes que passaram
    fp = int(((y_true == 0) & (y_pred == 1)).sum())  # legítimas bloqueadas
    return float(amount[fn].sum() + fp * cost_fp)


def threshold_min_custo(y_val, p_val, amount_val, cost_fp: float) -> float:
    """Varre 512 quantis dos scores DA VALIDAÇÃO e devolve o threshold
    que minimiza o custo esperado. Quantis (e não uma grade fixa) porque
    a distribuição dos scores é extremamente concentrada perto de zero."""
    candidatos = np.unique(np.quantile(p_val, np.linspace(0, 1, 512)))
    custos = [custo_esperado(y_val, (p_val >= t).astype(int), amount_val, cost_fp)
              for t in candidatos]
    return float(candidatos[int(np.argmin(custos))])


# Referências econômicas triviais no TESTE (calculadas agora, usadas só na Seção 10):
CUSTO_APROVAR_TUDO = float(amt_te[y_te == 1].sum())          # nenhum bloqueio
CUSTO_BLOQUEAR_TUDO = float((y_te == 0).sum() * CONFIG["cost_fp"])
print(f"Referências no teste -> aprovar tudo: R${CUSTO_APROVAR_TUDO:,.0f} | "
      f"bloquear tudo: R${CUSTO_BLOQUEAR_TUDO:,.0f}")
print("Qualquer modelo que não bater 'aprovar tudo' DESTRÓI valor.")

## 8. Inferência estatística: três incertezas diferentes

A v1 reportava números pontuais ("0,8247 venceu 0,7800") sem nenhuma medida de incerteza — com ~90 fraudes no teste, essa diferença cabe dentro do ruído amostral. Aqui separamos **três fontes de variação**, cada uma com sua ferramenta:

1. **Variabilidade algorítmica** (entre seeds): florestas, boosting e *qualquer configuração com SMOTE* (que sorteia vizinhos) mudam com a semente. Rodamos `n_seeds` execuções e reportamos **média ± desvio** do AP. As *decisões* usam a **média das probabilidades** entre seeds (um *seed-ensemble*, preditor mais estável que qualquer seed isolada).
2. **Variabilidade amostral do teste**: **bootstrap estratificado** (EFRON; TIBSHIRANI, 1993) — reamostramos o teste com reposição preservando o nº de fraudes e obtemos o **IC 95%** do AP.
3. **Comparação pareada entre modelos**: **teste de McNemar exato** (DIETTERICH, 1998) — cada transação foi classificada pelos dois modelos; olhamos só as *discordâncias* (b = só A erra, c = só B erra) e testamos H₀: b,c ~ Binomial(b+c, ½). Se b+c é pequeno, não há poder para distinguir os modelos — e dizer isso é um resultado honesto.

**Métrica primária = PR-AUC (Average Precision)**, não ROC-AUC: com prevalência de 0,17%, o FPR da curva ROC é diluído por ~284 mil negativos e a ROC-AUC fica inflada (SAITO; REHMSMEIER, 2015). A própria v1 exibia o sintoma sem diagnosticá-lo: o Naive Bayes tinha a *melhor* ROC-AUC do baseline (0,9342) e terminou com o *pior* F1 (0,3261).

In [ ]:
def bootstrap_ci(metric_fn, y, p, n_boot: int, seed: int):
    """IC 95% por bootstrap ESTRATIFICADO: reamostra positivos e negativos
    separadamente, preservando o número de fraudes em cada réplica."""
    rng = np.random.default_rng(seed)
    idx_pos = np.flatnonzero(y == 1)
    idx_neg = np.flatnonzero(y == 0)
    stats = []
    for _ in range(n_boot):
        bi = np.concatenate([rng.choice(idx_pos, len(idx_pos), replace=True),
                             rng.choice(idx_neg, len(idx_neg), replace=True)])
        stats.append(metric_fn(y[bi], p[bi]))
    lo, hi = np.percentile(stats, [2.5, 97.5])
    return round(float(lo), 4), round(float(hi), 4)


def mcnemar_exato(y_true, pred_a, pred_b):
    """Teste de McNemar exato (binomial) sobre as DISCORDÂNCIAS de erro
    entre dois classificadores no MESMO conjunto de teste (DIETTERICH, 1998)."""
    err_a = pred_a != y_true
    err_b = pred_b != y_true
    b = int((err_a & ~err_b).sum())   # transações em que SÓ A erra
    c = int((~err_a & err_b).sum())   # transações em que SÓ B erra
    if b + c == 0:
        return b, c, 1.0              # modelos idênticos nas discordâncias
    p = float(binomtest(min(b, c), n=b + c, p=0.5).pvalue)
    return b, c, p


def seeds_para(nome: str, estrategia: str) -> list:
    """Configuração é estocástica se o MODELO tem aleatoriedade OU se a
    estratégia é SMOTE (o sorteio de vizinhos torna estocástico até um
    modelo determinista). Deterministas rodam 1 vez (economia de tempo)."""
    estocastica = (nome not in DETERMINISTIC) or (estrategia == "smote")
    k = CONFIG["n_seeds"] if estocastica else 1
    return [CONFIG["base_seed"] + i for i in range(k)]


print("Ferramentas de inferência prontas (bootstrap, McNemar, política de seeds).")

## 9. Tuning do XGBoost — seleção **na validação**

O notebook da v1 anunciava no cabeçalho *"Validação Cruzada — para encontrar melhores parâmetros"*, importava `GridSearchCV`... e nunca o chamava. Os hiperparâmetros eram valores fixos com comentários racionalizando a posteriori.

Aqui o protocolo correto, com orçamento deliberadamente pequeno (3 candidatos): treinar no **treino**, comparar AP na **validação**, escolher — o teste permanece intocado (CAWLEY; TALBOT, 2010). A troca fundamental de hiperparâmetros do boosting: `n_estimators` × `learning_rate` se compensam (mais árvores ⇔ passo menor) e `max_depth` controla a ordem das interações entre features (FRIEDMAN, 2001). A extensão natural (v2.2) é busca aleatória com `TimeSeriesSplit` aninhado (BERGSTRA; BENGIO, 2012).

In [ ]:
if CONFIG["run_xgb_tuning"]:
    # Grade minúscula de propósito: demonstra o PROTOCOLO (seleção na validação),
    # não uma busca exaustiva. (n_estimators, learning_rate, max_depth)
    grade = [(300, 0.10, 4), (400, 0.10, 6), (400, 0.05, 8)]
    melhor, melhor_ap = None, -1.0
    print("Tuning do XGBoost (avaliado na VALIDAÇÃO — o teste segue intocado):")
    for n_est, lr, prof in grade:
        m = XGBClassifier(n_estimators=n_est, learning_rate=lr, max_depth=prof,
                          subsample=0.8, colsample_bytree=0.8,
                          tree_method="hist", eval_metric="aucpr",
                          n_jobs=-1, random_state=CONFIG["base_seed"])
        m.fit(X_tr, y_tr)                                   # ajusta no TREINO
        ap = average_precision_score(y_va, m.predict_proba(X_va)[:, 1])  # avalia na VAL
        print(f"  n={n_est:3d}  lr={lr:.2f}  depth={prof}  ->  AP_val = {ap:.4f}")
        if ap > melhor_ap:
            melhor_ap, melhor = ap, (n_est, lr, prof)
    # Congela a escolha na CONFIG — a Seção 6 lê daqui
    CONFIG["_xgb_n_estimators"], CONFIG["_xgb_lr"], CONFIG["_xgb_depth"] = melhor
    print(f"Escolhido: n={melhor[0]}, lr={melhor[1]}, depth={melhor[2]} "
          f"(AP_val={melhor_ap:.4f})")

## 10. Experimento principal

**A única célula que toca o teste.** O laço percorre 7 modelos × 3 estratégias (+ 2 não supervisionados), repete cada configuração estocástica em `n_seeds` sementes e, para cada configuração:

1. calcula AP no teste **por seed** (→ média ± desvio: instabilidade algorítmica);
2. tira a **média das probabilidades** entre seeds (seed-ensemble) para as decisões;
3. escolhe `thr_f1` e `thr_custo` **na validação** sobre essa média;
4. aplica os thresholds congelados **uma única vez** no teste;
5. calcula IC 95% por bootstrap, Brier (calibração) e o **custo em R$**.

Ao final, o McNemar compara as duas melhores configurações pelo critério econômico.

> Tempo estimado no dataset completo: 1,5–4 h (gargalos: Random Forest+SMOTE e o autoencoder). Para reduzir: `n_seeds=3` ou `sample_n=100_000` — **decidido antes de olhar qualquer resultado**.

In [ ]:
@dataclass
class Linha:
    """Uma linha da tabela de resultados (uma configuração modelo x estratégia)."""
    model: str; strategy: str; n_seeds: int
    ap_val: float                      # AP na validação (usada p/ ESCOLHER, não p/ reportar)
    ap_test_mean: float; ap_test_std: float   # média±desvio entre seeds (teste)
    ap_test_ci: tuple                  # IC 95% bootstrap (teste)
    roc_test: float                    # secundária (inflada em desbalanceamento)
    brier_test: object                 # calibração (None p/ não supervisionados)
    thr_f1: float; f1_test: float; prec_test: float; rec_test: float
    thr_cost: float; cost_test: float  # decisão econômica
    fit_seconds: float


linhas, preds_custo, probas_teste = [], {}, {}


def consolidar(nome, estrategia, aps_te, p_va_m, p_te_m, seg, brier_ok):
    """Fecha uma configuração: thresholds na VALIDAÇÃO, avaliação no TESTE."""
    # --- decisões: SOMENTE com dados de validação -------------------------
    thr_f1 = threshold_max_f1(y_va, p_va_m)
    thr_c  = threshold_min_custo(y_va, p_va_m, amt_va, CONFIG["cost_fp"])
    # --- aplicação única no teste, thresholds congelados -------------------
    y_hat_f1 = (p_te_m >= thr_f1).astype(int)
    y_hat_c  = (p_te_m >= thr_c ).astype(int)

    linha = Linha(
        model=nome, strategy=estrategia, n_seeds=len(aps_te),
        ap_val=round(float(average_precision_score(y_va, p_va_m)), 4),
        ap_test_mean=round(float(np.mean(aps_te)), 4),
        ap_test_std=round(float(np.std(aps_te)), 4),
        ap_test_ci=bootstrap_ci(average_precision_score, y_te, p_te_m,
                                CONFIG["n_bootstrap"], CONFIG["base_seed"]),
        roc_test=round(float(roc_auc_score(y_te, p_te_m)), 4),
        brier_test=(round(float(brier_score_loss(y_te, p_te_m)), 6)
                    if brier_ok else None),
        thr_f1=round(thr_f1, 4),
        f1_test=round(float(f1_score(y_te, y_hat_f1)), 4),
        prec_test=round(float(precision_score(y_te, y_hat_f1, zero_division=0)), 4),
        rec_test=round(float(recall_score(y_te, y_hat_f1)), 4),
        thr_cost=round(thr_c, 4),
        cost_test=round(custo_esperado(y_te, y_hat_c, amt_te, CONFIG["cost_fp"]), 2),
        fit_seconds=round(seg, 1))
    linhas.append(linha)

    chave = f"{nome} | {estrategia}"
    preds_custo[chave] = y_hat_c          # p/ McNemar
    probas_teste[chave] = p_te_m          # p/ gráficos
    print(f"[{chave:42s}] AP={linha.ap_test_mean:.4f}±{linha.ap_test_std:.4f} "
          f"F1={linha.f1_test:.4f} custo=R${linha.cost_test:,.0f} "
          f"({linha.n_seeds} seed(s), {seg:.0f}s)")


# ===================== SUPERVISIONADOS: modelo x estratégia x seeds =====================
for nome in SUPERVISED:
    for estrategia in STRATEGIES:
        t0 = time.time()
        aps, pv, pt = [], [], []
        for sd in seeds_para(nome, estrategia):
            m = make_supervised(nome, estrategia, sd)
            m.fit(X_tr, y_tr)                              # ajusta SÓ no treino
            pv.append(m.predict_proba(X_va)[:, 1])         # scores p/ decisões
            p_te = m.predict_proba(X_te)[:, 1]
            pt.append(p_te)
            aps.append(average_precision_score(y_te, p_te))  # AP por seed
        consolidar(nome, estrategia, aps,
                   np.mean(pv, axis=0), np.mean(pt, axis=0),  # seed-ensemble
                   time.time() - t0, brier_ok=True)

# ===================== NÃO SUPERVISIONADOS: modelo x seeds ==============================
for nome in UNSUPERVISED:
    t0 = time.time()
    aps, pv, pt = [], [], []
    for sd in seeds_para(nome, "unsupervised"):
        scorer = IsoForestScorer(sd) if nome == "Isolation Forest" else AutoencoderScorer(sd)
        scorer.fit(X_tr, y_tr)                 # AE usa y só p/ FILTRAR legítimas
        s_va, s_te = scorer.score(X_va), scorer.score(X_te)
        pv.append(minmax_to_proba(s_va, s_va))
        pt.append(minmax_to_proba(s_va, s_te)) # normalização pelos limites da VALIDAÇÃO
        aps.append(average_precision_score(y_te, pt[-1]))
    consolidar(nome, "unsupervised", aps,
               np.mean(pv, axis=0), np.mean(pt, axis=0),
               time.time() - t0, brier_ok=False)

# ===================== TABELA + McNEMAR =================================================
df_res = pd.DataFrame([asdict(l) for l in linhas])
df_res = df_res.sort_values(["cost_test", "ap_test_mean"],
                            ascending=[True, False]).reset_index(drop=True)
df_res.to_csv(OUTDIR / "results_v2_1.csv", index=False)

top2 = df_res.nsmallest(2, "cost_test")
k1 = f"{top2.iloc[0]['model']} | {top2.iloc[0]['strategy']}"
k2 = f"{top2.iloc[1]['model']} | {top2.iloc[1]['strategy']}"
b, c, pval = mcnemar_exato(y_te, preds_custo[k1], preds_custo[k2])
json.dump({"config_a": k1, "config_b": k2, "only_a_errs": b,
           "only_b_errs": c, "p_value": round(pval, 4)},
          open(OUTDIR / "mcnemar.json", "w"), indent=2)

print(f"\nMcNemar  {k1}  vs  {k2}:  b={b}, c={c}, p={pval:.4f}")
print("(p alto = indistinguíveis com estes dados — conclusão honesta, não fracasso)\n")

cols = ["model", "strategy", "n_seeds", "ap_test_mean", "ap_test_std",
        "ap_test_ci", "roc_test", "brier_test", "f1_test", "thr_cost", "cost_test"]
df_res[cols]

## 11. Leitura da tabela

- **`ap_test_mean ± ap_test_std`** — desempenho de ranqueamento e instabilidade entre seeds. Compare com **`ap_test_ci`**: se os ICs de duas configurações se sobrepõem fortemente, a diferença pontual não sustenta conclusão.
- **`brier_test`** — quanto menor, melhor calibrada a probabilidade. Repare como as estratégias `class_weight`/`smote` tendem a *piorar* o Brier: elas distorcem a prevalência efetiva de treino — o tema do paper que dá nome ao dataset (DAL POZZOLO et al., 2015).
- **`cost_test`** — a coluna que ordena a tabela: perda total em R$ com o threshold econômico. Compare com as referências `aprovar tudo` e `bloquear tudo` impressas na Seção 7.
- **Resultado esperado no dataset real**: `baseline` ≥ `class_weight` ≥ `smote` quando o threshold já é otimizado (VAN DEN GOORBERGH et al., 2022; ELOR; AVERBUCH-ELOR, 2022). Se confirmado, esse é o **achado central da monografia** — a v1 apontava para ele, mas o desenho confundido não permitia afirmá-lo.
- Não supervisionados terão AP muito menor: o valor deles é robustez a padrões *novos* (drift), não competir no padrão conhecido — reporte-os como classe separada.

## 12. Análise de sensibilidade da razão de custos

`cost_fp` é uma **estimativa** — quanto custa exatamente uma revisão manual? O resultado precisa ser robusto a variações plausíveis, o análogo direto da análise de sensibilidade de avaliação de projetos. Para as 3 melhores configurações, reescolhemos o threshold **na validação** para cada custo da grade e recalculamos o custo no teste.

**Se o ranking de modelos mudar com `cost_fp`, isso é um resultado, não um problema**: a escolha do classificador é uma decisão econômica que depende da estrutura de custos da instituição — argumento perfeito para a seção de Discussão.

In [ ]:
linhas_sens = []
for _, r in df_res.nsmallest(3, "cost_test").iterrows():
    nome, estrat = r["model"], r["strategy"]

    # Reajusta a configuração com a seed-base p/ obter scores de validação e teste
    if estrat == "unsupervised":
        scorer = (IsoForestScorer(CONFIG["base_seed"]) if nome == "Isolation Forest"
                  else AutoencoderScorer(CONFIG["base_seed"]))
        scorer.fit(X_tr, y_tr)
        s_va = scorer.score(X_va)
        p_va = minmax_to_proba(s_va, s_va)
        p_te = minmax_to_proba(s_va, scorer.score(X_te))
    else:
        m = make_supervised(nome, estrat, CONFIG["base_seed"])
        m.fit(X_tr, y_tr)
        p_va = m.predict_proba(X_va)[:, 1]
        p_te = m.predict_proba(X_te)[:, 1]

    # Para cada custo de FP da grade: threshold na VALIDAÇÃO, custo no TESTE
    for cfp in CONFIG["cost_fp_grid"]:
        t = threshold_min_custo(y_va, p_va, amt_va, cfp)
        c = custo_esperado(y_te, (p_te >= t).astype(int), amt_te, cfp)
        linhas_sens.append({"config": f"{nome} | {estrat}", "cost_fp": cfp,
                            "threshold": round(t, 4), "cost_test": round(c, 2)})

sens = pd.DataFrame(linhas_sens)
sens.to_csv(OUTDIR / "cost_sensitivity.csv", index=False)

# Pivot: linhas = configuração, colunas = custo de FP, células = custo total no teste
sens.pivot(index="config", columns="cost_fp", values="cost_test").round(0)

## 13. Visualizações

Três gráficos, cada um respondendo a uma pergunta:

1. **Curvas Precision–Recall no teste** — *o modelo ordena bem as fraudes?* A linha tracejada é o baseline aleatório (= prevalência): toda a área entre ela e a curva é o valor agregado pelo modelo. (Na curva ROC, esse baseline seria a diagonal — visualmente enganosa sob desbalanceamento.)
2. **Curvas de calibração** — *a probabilidade significa o que diz?* Pontos sobre a diagonal = calibração perfeita; pré-requisito para o threshold econômico ter interpretação (Seção 7).
3. **Custo × threshold do melhor modelo** — *a decisão econômica em uma imagem*: o vale da curva é a operação ótima; a linha vermelha marca o threshold escolhido **na validação** (repare se ele cai perto do vale do teste — é o sinal de que a escolha generalizou); a linha cinza é o custo de "aprovar tudo".

In [ ]:
# --- 1) Curvas PR no teste (uma estratégia por modelo, ESCOLHIDA PELA VALIDAÇÃO) ---
melhor_por_modelo = df_res.loc[df_res.groupby("model")["ap_val"].idxmax()]

fig, ax = plt.subplots(figsize=(9, 6))
for _, r in melhor_por_modelo.iterrows():
    chave = f"{r['model']} | {r['strategy']}"
    prec, rec, _ = precision_recall_curve(y_te, probas_teste[chave])
    ax.plot(rec, prec, label=f"{r['model']} (AP={r['ap_test_mean']:.3f})")
ax.axhline(y_te.mean(), ls="--", c="gray", lw=1,
           label=f"aleatório (AP = prevalência = {y_te.mean():.4f})")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Precision–Recall no teste (estratégia escolhida na validação)")
ax.legend(fontsize=7); fig.tight_layout()
fig.savefig(OUTDIR / "pr_curves_test.png", dpi=150); plt.show()

# --- 2) Calibração dos 3 melhores supervisionados -------------------------------
fig, ax = plt.subplots(figsize=(7, 6))
sup = melhor_por_modelo[melhor_por_modelo["strategy"] != "unsupervised"]
for _, r in sup.nlargest(3, "ap_val").iterrows():
    chave = f"{r['model']} | {r['strategy']}"
    CalibrationDisplay.from_predictions(y_te, probas_teste[chave],
                                        n_bins=10, ax=ax, name=r["model"])
ax.set_title("Curvas de calibração no teste")
fig.tight_layout(); fig.savefig(OUTDIR / "calibration_test.png", dpi=150); plt.show()

# --- 3) Custo x threshold do campeão econômico ----------------------------------
topo = df_res.nsmallest(1, "cost_test").iloc[0]
chave = f"{topo['model']} | {topo['strategy']}"
p = probas_teste[chave]
grade_t = np.linspace(0, 1, 200)
curva_custo = [custo_esperado(y_te, (p >= t).astype(int), amt_te,
                              CONFIG["cost_fp"]) for t in grade_t]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(grade_t, curva_custo)
ax.axvline(topo["thr_cost"], ls="--", c="red",
           label=f"threshold escolhido NA VALIDAÇÃO = {topo['thr_cost']:.3f}")
ax.axhline(CUSTO_APROVAR_TUDO, ls=":", c="gray",
           label=f"aprovar tudo = R${CUSTO_APROVAR_TUDO:,.0f}")
ax.set_xlabel("Threshold"); ax.set_ylabel("Custo total no teste (R$)")
ax.set_title(f"Custo econômico x threshold — {chave}")
ax.legend(fontsize=8); fig.tight_layout()
fig.savefig(OUTDIR / "cost_vs_threshold.png", dpi=150); plt.show()

## 14. Interpretabilidade com SHAP (opcional)

A v1 usava "interpretabilidade" como argumento para a Árvore de Decisão — e depois a destruía aplicando PCA sobre todas as variáveis. O SHAP (LUNDBERG; LEE, 2017) desfaz esse trade-off: atribui a cada feature, **em cada transação individual**, uma contribuição aditiva baseada nos **valores de Shapley** da teoria dos jogos cooperativos (SHAPLEY, 1953 — teoria de economista). Resultado: pode-se usar o modelo mais forte (XGBoost) *e* explicar cada alerta ao analista que o revisa — o requisito operacional e regulatório real de um sistema antifraude.

O `TreeExplainer` calcula os valores de forma exata e eficiente para modelos de árvore. A célula é tolerante: se o `shap` não estiver instalado, apenas avisa e segue.

In [ ]:
if CONFIG["run_shap"]:
    try:
        import shap
    except ImportError:
        shap = None
        print("Pacote 'shap' não instalado — etapa pulada (Seção 0 instala).")

    if shap is not None:
        # Reajusta o XGBoost com os hiperparâmetros escolhidos na Seção 9
        modelo_shap = XGBClassifier(
            n_estimators=CONFIG.get("_xgb_n_estimators", 400),
            learning_rate=CONFIG.get("_xgb_lr", 0.1),
            max_depth=CONFIG.get("_xgb_depth", 6),
            subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=POS_RATIO,           # versão reponderada: alertas com recall alto
            tree_method="hist", eval_metric="aucpr",
            n_jobs=-1, random_state=CONFIG["base_seed"])
        modelo_shap.fit(X_tr, y_tr)

        # Amostra do teste (2.000 obs): SHAP exato em 43k linhas seria lento sem ganho
        rng = np.random.default_rng(CONFIG["base_seed"])
        idx = rng.choice(len(X_te), size=min(2000, len(X_te)), replace=False)

        explainer = shap.TreeExplainer(modelo_shap)      # exato p/ árvores
        valores = explainer.shap_values(X_te[idx])

        shap.summary_plot(valores, X_te[idx], feature_names=FEATURES,
                          max_display=15, show=False)
        plt.tight_layout()
        plt.savefig(OUTDIR / "shap_summary_xgb.png", dpi=150, bbox_inches="tight")
        plt.show()
        print("Leitura: cada ponto é UMA transação; cor = valor da feature;",
              "posição horizontal = contribuição para o score de fraude.")

## 15. Como reportar na monografia + Referências

**Estrutura sugerida da seção de Resultados:**
1. Tabela principal: `ap_test_mean ± std` e `ap_test_ci` por modelo × estratégia (a incerteza acompanha *todo* número comparado).
2. O achado da ablação: efeito isolado de class_weight e SMOTE vs baseline, com a discussão de calibração (Brier).
3. O resultado econômico: `cost_test` vs "aprovar tudo" — economia em R$ no período de teste — e a tabela de sensibilidade (Seção 12), discutindo inversões de ranking se houver.
4. McNemar entre os dois melhores: afirmar diferença **apenas** se p < 0,05; caso contrário, reportar equivalência estatística (conclusão legítima).
5. Não supervisionados como classe à parte: papel de robustez a drift, não de competidor direto.
6. SHAP como resposta ao requisito de interpretabilidade (substitui o argumento genérico da v1).

---

### Referências desta implementação (formato ABNT aproximado)

BAHNSEN, A. C.; AOUADA, D.; OTTERSTEN, B. Example-dependent cost-sensitive logistic regression for credit scoring. In: **ICMLA**, 2014.

BERGSTRA, J.; BENGIO, Y. Random search for hyper-parameter optimization. **JMLR**, v. 13, 2012.

BREIMAN, L. Random forests. **Machine Learning**, v. 45, n. 1, 2001.

CAWLEY, G. C.; TALBOT, N. L. C. On over-fitting in model selection and subsequent selection bias in performance evaluation. **JMLR**, v. 11, 2010.

CHAWLA, N. V. et al. SMOTE: Synthetic Minority Over-sampling Technique. **JAIR**, v. 16, 2002.

CHEN, T.; GUESTRIN, C. XGBoost: a scalable tree boosting system. In: **KDD**, 2016.

DAL POZZOLO, A. **Adaptive machine learning for credit card fraud detection**. Tese (Doutorado) — Université Libre de Bruxelles, 2015.

DAL POZZOLO, A. et al. Calibrating probability with undersampling for unbalanced classification. In: **IEEE SSCI**, 2015. *(citação correta do dataset)*

DIETTERICH, T. G. Approximate statistical tests for comparing supervised classification learning algorithms. **Neural Computation**, v. 10, n. 7, 1998.

EFRON, B.; TIBSHIRANI, R. J. **An introduction to the bootstrap**. Chapman & Hall, 1993.

ELKAN, C. The foundations of cost-sensitive learning. In: **IJCAI**, 2001.

ELOR, Y.; AVERBUCH-ELOR, H. To SMOTE, or not to SMOTE? **arXiv:2201.08528**, 2022.

FRIEDMAN, J. H. Greedy function approximation: a gradient boosting machine. **Annals of Statistics**, v. 29, n. 5, 2001.

GAMA, J. et al. A survey on concept drift adaptation. **ACM Computing Surveys**, v. 46, n. 4, 2014.

GOODFELLOW, I.; BENGIO, Y.; COURVILLE, A. **Deep learning**. MIT Press, 2016. cap. 14.

HE, H.; GARCIA, E. A. Learning from imbalanced data. **IEEE TKDE**, v. 21, n. 9, 2009.

LIU, F. T.; TING, K. M.; ZHOU, Z.-H. Isolation forest. In: **ICDM**, 2008.

LUNDBERG, S. M.; LEE, S.-I. A unified approach to interpreting model predictions. In: **NeurIPS**, 2017.

PLATT, J. Probabilistic outputs for support vector machines. In: **Advances in Large Margin Classifiers**, 1999.

SAITO, T.; REHMSMEIER, M. The precision-recall plot is more informative than the ROC plot when evaluating binary classifiers on imbalanced datasets. **PLOS ONE**, v. 10, n. 3, 2015.

SHAPLEY, L. S. A value for n-person games. In: **Contributions to the Theory of Games II**, 1953.

VAN DEN GOORBERGH, R. et al. The harm of class imbalance corrections for risk prediction models. **JAMIA**, v. 29, n. 9, 2022.

WILLIAMSON, O. E. Transaction cost economics: how it works; where it is headed. **De Economist**, v. 146, 1998.
